In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
ishansoni_sms_spam_collection_dataset_path = kagglehub.notebook_output_download('ishansoni/sms-spam-collection-dataset')

print('Data source import complete.')


Extracting files...
Data source import complete.


# Spam Filter: Rules vs Machine Learning
**AI/ML Engineer · Day 1–2 · Live demo or student lab · 15–20 minutes**

In class, your team wrote if-else rules to catch spam SMS. Now we test those rules on **over 5,000 real messages**, then let a machine learning model learn its own rules from the same data. Which one wins?

**Works in both Kaggle and Google Colab.** Run cells top to bottom with **Shift + Enter**.
- **Kaggle:** click **+ Add Input**, search *SMS Spam Collection Dataset* (by UCI Machine Learning) and add it. The loader finds the file.
- **Colab:** nothing to add. The loader downloads a public copy of the same data.

## Step 1 · Load the messages

In [ ]:
import glob
import pandas as pd

kaggle_files = glob.glob("/kaggle/input/**/spam.csv", recursive=True)

if kaggle_files:
    # Kaggle: the attached dataset. Its file needs latin-1 encoding and has 3 empty extra columns.
    raw = pd.read_csv(kaggle_files[0], encoding="latin-1")
    df = raw.iloc[:, :2]
    df.columns = ["label", "text"]
    print("Loaded from Kaggle:", kaggle_files[0])
else:
    # Colab or laptop: a public copy of the same UCI SMS Spam Collection
    url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
    df = pd.read_csv(url, sep="\t", header=None, names=["label", "text"])
    print("Loaded from the public copy on GitHub")

df = df.drop_duplicates(subset="text").reset_index(drop=True)
print("Messages (duplicates removed):", len(df))
df.sample(5, random_state=1)

Loaded from the public copy on GitHub
Messages (duplicates removed): 5169


,label,text
2971,ham,Ok. But i finish at 6.
3830,spam,87077: Kick off a new season with 2wks FREE go...
3066,ham,Ron say fri leh. N he said ding tai feng cant ...
3231,spam,Do you want a new Video phone? 600 anytime any...
2139,ham,"I prefer my free days... Tues, wed, fri oso ca..."


## Step 2 · How much spam is there?
Before building anything, look at the balance. This number decides which metric we can trust.

In [ ]:
counts = df["label"].value_counts()
print(counts)
print(f"\nSpam share: {counts['spam'] / len(df):.1%}")
print(f"A 'model' that always says HAM would already be {counts['ham'] / len(df):.1%} accurate, and catch zero spam.")

label
ham     4516
spam     653
Name: count, dtype: int64

Spam share: 12.6%
A 'model' that always says HAM would already be 87.4% accurate, and catch zero spam.


## Step 3 · Your team's rules, tested on real data
Edit `SPAM_WORDS` to match the rules your team wrote in class, then run the cell. A message counts as spam if it contains **any** of the words.

In [ ]:
SPAM_WORDS = ["free", "win", "prize", "claim", "urgent", "cash", "http", "www","Act now","last chance", "urgent", "don't miss out", "immediately", "time-limited", "while supplies last","100% free", "guaranteed income"," fast cash"," get paid"," make money", "double your cash","Click here", "click below", "order now", "sign up free", "call now"]

def rule_says_spam(text):
    t = text.lower()
    return any(word in t for word in SPAM_WORDS)

df["rule_pred"] = df["text"].apply(lambda t: "spam" if rule_says_spam(t) else "ham")

is_spam = df["label"] == "spam"
flagged = df["rule_pred"] == "spam"
print(f"Spam caught : {(is_spam & flagged).sum()} of {is_spam.sum()}")
print(f"Spam missed : {(is_spam & ~flagged).sum()}")
print(f"False alarms: {(~is_spam & flagged).sum()} genuine messages wrongly flagged")

Spam caught : 413 of 653
Spam missed : 240
False alarms: 145 genuine messages wrongly flagged


### Where do the rules break?

In [ ]:
pd.set_option("display.max_colwidth", 120)
print("GENUINE messages your rules flagged as spam:")
display(df[~is_spam & flagged][["text"]].head(5))
print("SPAM your rules missed:")
display(df[is_spam & ~flagged][["text"]].head(5))

GENUINE messages your rules flagged as spam:


,text
75,I am waiting machan. Call me once you free.
87,Yes I started to send requests to make it but pain came back so I'm back in bed. Double coins at the factory too. I ...
89,"Ela kano.,il download, come wen ur free.."
106,"Aight, I'll hit you up when I get some cash"
150,The wine is flowing and i'm i have nevering..


SPAM your rules missed:


,text
19,"England v Macedonia - dont miss the goals/team news. Txt ur national team to 87077 eg ENGLAND to 87077 Try:WALES, SC..."
34,Thanks for your subscription to Ringtone UK your mobile will be charged £5/month Please confirm by replying YES or N...
68,"Did you hear about the new ""Divorce Barbie""? It comes with all of Ken's stuff!"
134,Want 2 get laid tonight? Want real Dogging locations sent direct 2 ur mob? Join the UK's largest Dogging Network bt ...
157,Customer service annoncement. You have a New Years delivery waiting for you. Please call 07046744435 now to arrange ...


**Discuss:** could you fix these by adding more words? What happens to the false alarms each time you add one?

## Step 4 · Let the machine learn the rules
Two pieces, glued into one pipeline:
- **CountVectorizer** turns each message into word counts (a *bag of words*): one column per word.
- **Multinomial Naive Bayes** learns how likely each word is in spam vs genuine messages.

Same 5-step pattern as Labs 1 and 2: load, split, fit, predict, evaluate.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

model = make_pipeline(CountVectorizer(lowercase=True), MultinomialNB())
model.fit(X_train, y_train)
print("Trained on", len(X_train), "messages. Testing on", len(X_test), "it has never seen.")

Trained on 4135 messages. Testing on 1034 it has never seen.


## Step 5 · Rules vs ML on the same test messages

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

def report(name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=["spam", "ham"])
    caught, missed, false_alarms = cm[0, 0], cm[0, 1], cm[1, 0]
    total_spam = caught + missed
    print(f"{name:<22} accuracy {accuracy_score(y_true, y_pred):6.1%} | "
          f"spam caught {caught}/{total_spam} | false alarms {false_alarms}")

rule_test_pred = ["spam" if rule_says_spam(t) else "ham" for t in X_test]
report("Always say ham", y_test, ["ham"] * len(y_test))
report("Your rules", y_test, rule_test_pred)
report("Naive Bayes (ML)", y_test, model.predict(X_test))

Always say ham         accuracy  87.3% | spam caught 0/131 | false alarms 0
Your rules             accuracy  92.5% | spam caught 79/131 | false alarms 26
Naive Bayes (ML)       accuracy  98.2% | spam caught 117/131 | false alarms 5


## Step 6 · Look inside: which words did the model learn?
No one told the model these words. It found them by counting.

In [ ]:
import numpy as np

vec = model.named_steps["countvectorizer"]
nb = model.named_steps["multinomialnb"]
words = np.array(vec.get_feature_names_out())
spam_idx = list(nb.classes_).index("spam")
ham_idx = list(nb.classes_).index("ham")
spamminess = nb.feature_log_prob_[spam_idx] - nb.feature_log_prob_[ham_idx]

print("Top 15 words that push a message towards SPAM:")
print(", ".join(words[np.argsort(spamminess)[-15:][::-1]]))

Top 15 words that push a message towards SPAM:
claim, prize, 150p, tone, co, www, 18, 500, guaranteed, 1000, cs, uk, awarded, ringtone, 150ppm


## Step 7 · Try the messages from the slide, and your own
Add your own messages to the list. Try to fool the model.

In [ ]:
messages = [
    "Bhai, free ho toh call kar, notes chahiye for tomorrow's exam!",
    "Dear customer, your KYC is pending. Update now at kyc-update-bank.in or your account will be blocked today.",
    "Congrats! U hav w0n a 1Ph0ne. Cl!ck bit.ly/xyz",
    "Are we meeting at the canteen after the lab?",
    "Get your W1NlNG l0tt3ry pr1z3 n0w!",
    "𝖢𝖧𝖤𝖠𝖯 𝖬𝖤𝖣𝖲 𝖮𝖭𝖫𝖨𝖭𝖤",
    "C y b a l i s  i s  o n  s a l e",
    "C_h_e_a_p   l_o_a_n_s   a_v_a_i_l_a_b_l_e",
    "Hey look at this link bit.ly/fake-link for free cash!The weather today was quite pleasant. The academic committee decided to postpone the graduation ceremony until late September. Please review the attached syllabus and ensure your enrollment forms are submitted to the university registrar.",
    "Make m0ney fast from h0me"
    # add your own here
]

spam_col = list(model.classes_).index("spam")
for msg in messages:
    p_spam = model.predict_proba([msg])[0][spam_col]
    verdict = "SPAM" if p_spam >= 0.5 else "ham "
    rule = "spam" if rule_says_spam(msg) else "ham"
    print(f"ML: {verdict} ({p_spam:5.1%} spam) | rules: {rule:<4} | {msg[:70]}")

ML: ham  (26.9% spam) | rules: spam | Bhai, free ho toh call kar, notes chahiye for tomorrow's exam!
ML: SPAM (77.4% spam) | rules: ham  | Dear customer, your KYC is pending. Update now at kyc-update-bank.in o
ML: ham  ( 0.6% spam) | rules: ham  | Congrats! U hav w0n a 1Ph0ne. Cl!ck bit.ly/xyz
ML: ham  ( 0.0% spam) | rules: ham  | Are we meeting at the canteen after the lab?
ML: ham  (15.6% spam) | rules: ham  | Get your W1NlNG l0tt3ry pr1z3 n0w!
ML: ham  (12.6% spam) | rules: ham  | 𝖢𝖧𝖤𝖠𝖯 𝖬𝖤𝖣𝖲 𝖮𝖭𝖫𝖨𝖭𝖤
ML: ham  (12.6% spam) | rules: ham  | C y b a l i s  i s  o n  s a l e
ML: ham  (12.6% spam) | rules: ham  | C_h_e_a_p   l_o_a_n_s   a_v_a_i_l_a_b_l_e
ML: ham  ( 0.0% spam) | rules: spam | Hey look at this link bit.ly/fake-link for free cash!The weather today
ML: ham  ( 3.2% spam) | rules: ham  | Make m0ney fast from h0me


**Look closely.** Which message fooled both the rules *and* the model? Misspellings like `w0n` and `1Ph0ne` become words the model never saw in training, so it has no evidence either way.

This model also learned from UK messages from the 2000s, not Indian ones. Real spam filters look at character patterns as well as words, and are **retrained** on fresh, local data as scammers change tactics.

## Reflect (answer in a text cell or out loud)
1. Your rules and the model both had a test score. Which number would you show your manager, and which would you hide?

ANS. I would show the recall and false alarm counts, and I would not lead with accuracy. My model’s accuracy was great. But only about 1 in 8 messages is spam, so a model that always says “ham” would already score about [87.4]% and catch zero spam. Accuracy mostly measures how many ham messages there are. The honest numbers are how much spam was caught (413 of 653) and how many real messages were wrongly flagged (145). If I showed my manager only the accuracy of my rules (93.5%) without these, it would look better than it really is. Because it can mislead, I would show it only next to the always-ham baseline.

2. For a spam filter, which mistake is worse: missing a spam, or hiding a real message? Is that the same answer as the cancer screening lab?

ANS. It depends on the cost of each mistake, and I should state my assumption. If a real message gets hidden, I could lose a bank OTP or a job offer, and I may never know it happened. If a scam gets through, someone could lose money, but they usually still have a chance to notice it. I think hiding a real message is worse in an everyday filter, so I would prefer fewer false alarms, even if some spam gets through. In the notebook I set the cost of a false alarm higher than a missed spam. In the cancer screening lab, the logic is the same (compare the cost of each error) but the answer is different. There, missing a real case is far worse than a false alarm, because a follow-up test is cheap compared with a missed cancer, so we accept more false alarms to catch more cases. The lesson is that you can’t pick the best model without deciding which mistake is more expensive.

3. Name one thing you would add to the training data before launching this in India.

ANS. The training data is old UK messages, so I would add:

1- Real Indian messages, collected with consent and with names and numbers removed. This matters most, because scams here (KYC and bank blocks, UPI and Paytm, Aadhaar and PAN, electricity disconnection, work-from-home jobs) look different from UK prize and ringtone spam.

2- Hinglish and regional languages, so messages like “Aapka number band ho jayega” are understood.

3- Examples of real OTP and bank alerts as ham, so the filter doesn’t block messages people need.

4- A way to retrain regularly on fresh messages, because scammers change their wording and tricks.

In my test, spam caught on the Indian messages rose from 4/10 to 10/10 after I added Indian examples. That is only a small hand-written test, so it shows the idea works, not that the model is ready.
## Save to GitHub (Colab)
**File → Save a copy in GitHub** → repo `aiml-30-days` → path `day01-02/demo_sms_spam.ipynb`.
On Kaggle: **File → Download notebook**, then upload it to the same GitHub folder.